# 02 — Retrieval Sanity Checks

Verifies FR-C1..C4: domain filtering works, top-k is configurable, sources are
preserved for citations, and **Arabic questions retrieve Arabic documents**
even though English near-matches exist in the same corpus.

Run `python -m src.vectorstore` first — this notebook needs a built index.

In [1]:
import sys
from pathlib import Path

# Works whether Jupyter was started from the repo root or from notebooks/:
# walk up until we find the package (starting `from src...` from the wrong
# cwd is the classic notebook failure).
here = Path.cwd()
root = next(p for p in [here, *here.parents] if (p / "src" / "settings.py").exists())
sys.path.insert(0, str(root))
print("project root:", root)

from src.retriever import retrieve, retrieve_multi_domain, unique_sources
from src.settings import settings

print("top-k =", settings.retrieval_top_k)

project root: /home/nourz/Desktop/Atlas_sprintsProject


top-k = 5


## Cross-language retrieval (FR-C4)

Queries taken from `tests/eval_cases.jsonl` — the Arabic ones must surface
Arabic or bilingual sources.

In [2]:
import json

cases = [
    json.loads(l)
    for l in settings.eval_cases_path.read_text(encoding="utf-8").splitlines()
    if l.strip()
]

for case in cases:
    if case["id"] not in {"EVAL-002", "EVAL-007", "EVAL-009"}:
        continue
    hits = retrieve_multi_domain(case["input"], list(settings.domains))
    print(f"\n### {case['id']} ({case['language']}) domain={case['expected_domain']}")
    print("Q:", case["input"][:100])
    print("expected sources:", case["expected_sources"])
    for h in hits[:5]:
        marker = "✓" if h.source_filename in case["expected_sources"] else " "
        print(f"  {marker} {h.score:.3f} [{h.domain}/{h.language}] {h.source_filename}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


### EVAL-002 (ar) domain=hr
Q: ما هو عدد أيام الإجازة المرضية المدفوعة سنويًا، ومتى يلزم تقديم شهادة طبية؟
expected sources: ['HR-007-سياسة-الإجازات-المرضية.md']
  ✓ 0.694 [hr/ar] HR-007-سياسة-الإجازات-المرضية.md
  ✓ 0.681 [hr/ar] HR-007-سياسة-الإجازات-المرضية.md
    0.609 [hr/en] HR-006-leave-types-guide.docx
    0.564 [hr/en] HR-006-leave-types-guide.docx
  ✓ 0.561 [hr/ar] HR-007-سياسة-الإجازات-المرضية.md



### EVAL-007 (en) domain=finance
Q: I traveled domestically for a client meeting. I paid for a hotel and a taxi from the airport. What's
expected sources: ['FIN-001-travel-reimbursement-policy.md', 'FIN-002-expense-claim-sop.pdf']
  ✓ 0.693 [finance/en] FIN-001-travel-reimbursement-policy.md
  ✓ 0.670 [finance/en] FIN-002-expense-claim-sop.pdf
  ✓ 0.661 [finance/en] FIN-001-travel-reimbursement-policy.md
  ✓ 0.637 [finance/en] FIN-001-travel-reimbursement-policy.md
    0.628 [finance/ar] FIN-007-سياسة-السفر-الدولي.md



### EVAL-009 (ar) domain=finance
Q: كم الحد الأقصى لتكلفة عشاء العملاء لكل شخص، ومتى أحتاج موافقة مسبقة؟
expected sources: ['FIN-009-عشاء-العملاء.docx']
  ✓ 0.701 [finance/ar] FIN-009-عشاء-العملاء.docx
  ✓ 0.689 [finance/ar] FIN-009-عشاء-العملاء.docx
  ✓ 0.669 [finance/ar] FIN-009-عشاء-العملاء.docx
    0.589 [finance/en] FIN-005-corporate-card-policy.pdf
    0.586 [finance/ar] FIN-007-سياسة-السفر-الدولي.md


## Domain filtering (FR-C1)

The same question retrieved with each single-domain filter should only
return chunks tagged with that domain.

In [3]:
q = "How many vacation days does a new hire get?"
for domain in settings.domains:
    hits = retrieve(q, domain, top_k=3)
    domains = {h.domain for h in hits}
    assert domains <= {domain}, f"filter leaked: {domains}"
    print(f"{domain:8s} -> {[h.source_filename for h in hits]}")
print("\nDomain filter: OK (FR-C1)")

hr       -> ['HR-001-vacation-policy.md', 'HR-003-onboarding-handbook.docx', 'HR-001-vacation-policy.md']


it       -> ['IT-003-laptop-provisioning.docx', 'IT-009-طلب-البرامج.docx', 'IT-003-laptop-provisioning.docx']


finance  -> ['FIN-002-expense-claim-sop.pdf', 'FIN-001-travel-reimbursement-policy.md', 'FIN-008-بدل-المواصلات.pdf']

Domain filter: OK (FR-C1)


## Multi-document questions

EVAL-007 expects **both** FIN-001 and FIN-002 — single-document retrieval
loses Contextual Recall points. Check the fan-out retrieval covers them.

In [4]:
case = next(c for c in cases if c["id"] == "EVAL-007")
hits = retrieve_multi_domain(case["input"], list(settings.domains), top_k=8)
retrieved = set(unique_sources(hits))
expected = set(case["expected_sources"])
print("expected:", sorted(expected))
print("retrieved:", sorted(retrieved))
print("missing:", sorted(expected - retrieved) or "none — full coverage ✓")

expected: ['FIN-001-travel-reimbursement-policy.md', 'FIN-002-expense-claim-sop.pdf']
retrieved: ['FIN-001-travel-reimbursement-policy.md', 'FIN-002-expense-claim-sop.pdf', 'FIN-007-سياسة-السفر-الدولي.md']
missing: none — full coverage ✓
